In [ ]:
import requests
import os
import json

# save file 

In [ ]:
url = "https://raw.githubusercontent.com/DataTalksClub/datatalksclub.github.io/refs/heads/main/_podcast/s24e06-how-to-build-ai-that-actually-ships-in-production.md"
response = requests.get(url)

In [ ]:
text = response.text

In [ ]:
path_cwd = os.getcwd()
path_cwd

In [ ]:
data_path = os.path.join(os.path.dirname(path_cwd),"data")

In [ ]:
data_path

In [ ]:
if not os.path.exists(data_path):
    os.makedirs(data_path)

In [ ]:
episode_path = os.path.join(data_path,"episode.md")

In [ ]:
with open(episode_path, "w",encoding="utf-8") as f:
    f.write(text)

# explore transcript

In [ ]:
response.text.split("transcript:")[1]

In [ ]:
response.text.split("transcript:")[1].split("\n- ")[2].split('line: ')

In [ ]:
response.text.split("transcript:")[1].split("\n- ")[2].split('line: ')[1].split('sec: ')[0]

In [ ]:
print(response.text.split("transcript:")[1].split("\n- ")[2].split('line: ')[1].split('sec: ')[0])

In [ ]:
episode_data = {}


### episode

In [ ]:
text.split('episode:')[1].split('\n')[0].strip()

In [ ]:
episode_data['episode'] = int(text.split('episode:')[1].split('\n')[0].strip())

### guests

In [ ]:
text.split('guests:\n-')[1].split('\n')[0].strip()

In [ ]:
episode_data['guest'] = text.split('guests:\n-')[1].split('\n')[0].strip()

### season

In [ ]:
text.split('season:')[1].split('\n')[0].strip()

In [ ]:
episode_data['season'] = int(text.split('season:')[1].split('\n')[0].strip())

### title

In [ ]:
text.split('title:')[1].split('\n')[0].strip()

In [ ]:
episode_data['title'] = text.split('title:')[1].split('\n')[0].strip()

### transcript

In [ ]:
print(text.split('transcript:\n')[1])

In [ ]:
headers_block = text.split('transcript:\n')[1].split('- header:')[1:]

In [ ]:
[{'header': i.split('\n')[0].strip() } for i in headers_block]

In [ ]:
episode_data['transcript'] = [{'header': i.split('\n')[0].strip() } for i in headers_block]

In [ ]:
print(headers_block[0])

In [ ]:
print(headers_block[0].split('- ')[0])

In [ ]:
print(headers_block[0].split('- ')[1])

In [ ]:
headers_block[0].split('- ')[1].split('line:')[1].split('sec:')[0].strip()

In [ ]:
headers_block[0].split('- ')[1].split('line:')[1].split('sec:')[1].split('\n')[0].strip()

In [ ]:
headers_block[0].split('- ')[1].split('line:')[1].split('time:')[1].split('\n')[0].strip()

In [ ]:
headers_block[0].split('- ')[1].split('line:')[1].split('who:')[1].split('\n')[0].strip()

In [ ]:
headers_block[0]

In [ ]:
[{'line': i.split('line:')[1].split('sec:')[0].strip(),
  'sec': i.split('line:')[1].split('sec:')[1].split('\n')[0].strip(),
  'time': i.split('line:')[1].split('time:')[1].split('\n')[0].strip(),
  'who': i.split('line:')[1].split('who:')[1].split('\n')[0].strip()} for i in headers_block[0].split('- ')[1:]]

In [ ]:
testing_lines = [{'header': i.split('\n')[0].strip(),
                               'lines': [{'line': j.split('line:')[1].split('sec:')[0].strip(),
                                            'sec': j.split('line:')[1].split('sec:')[1].split('\n')[0].strip(),
                                            'time': j.split('line:')[1].split('time:')[1].split('\n')[0].strip(),
                                            'who': j.split('line:')[1].split('who:')[1].split('\n')[0].strip()} 
                                            for j in i.split('- ')[1:]
                                        ] 
 } for i in headers_block]

In [ ]:
len(testing_lines)

In [ ]:
len(testing_lines[0])

In [ ]:
testing_lines[0].keys()

In [ ]:
[len(i['lines']) for i in testing_lines]

In [ ]:
episode_data['transcript'] = [{'header': i.split('\n')[0].strip(),
                               'lines': [{'line': j.split('line:')[1].split('sec:')[0].strip(),
                                          'sec': j.split('line:')[1].split('sec:')[1].split('\n')[0].strip(),
                                          'time': j.split('line:')[1].split('time:')[1].split('\n')[0].strip().strip("'"),
                                          'who': j.split('line:')[1].split('who:')[1].split('\n')[0].strip()} 
                                             for j in i.split('- ')[1:]
                                        ] 
                              } for i in headers_block]

## Functions

In [ ]:
def clean_markdown(input_text:str)->dict[str]:
    separators = {'episode':'episode:','guest':'guests:\n-','season':'season:','title':'title:'}
    output = { i: input_text.split(separators[i])[1].split('\n')[0].strip() for i in separators}
    output['episode'], output['season'] = int(output['episode']) , int(output['season'])
    
    headers_block = input_text.split('transcript:\n')[1].split('- header:')[1:]
    lines_separators = {'line':'sec:','sec':'sec:','time':'time:','who':'who:'}
    functions_dictionary = {'line':lambda x,y: x.split(y)[0].strip(),
                           'sec':lambda x,y: x.split(y)[1].split('\n')[0].strip(),
                           'time':lambda x,y: x.split(y)[1].split('\n')[0].strip().strip("'"),
                           'who':lambda x,y: x.split(y)[1].split('\n')[0].strip()
                          }
    output['transcript'] = [{'header': i.split('\n')[0].strip(),
                               'lines': [{k:functions_dictionary[k](j.split('line:')[1],lines_separators[k]) 
                                             for k in lines_separators}
                                             for j in i.split('- ')[1:]
                                        ] 
                              } for i in headers_block]
    return output

In [ ]:
lines_separators = {'line':'sec:','sec':'sec:','time':'time:','who':'who:'}
functions_dictionary = {'line':lambda x,y: x.split(y)[0].strip(),
                       'sec':lambda x,y: x.split(y)[1].split('\n')[0].strip(),
                       'time':lambda x,y: x.split(y)[1].split('\n')[0].strip().strip("'"),
                       'who':lambda x,y: x.split(y)[1].split('\n')[0].strip()
                      }

In [ ]:
{i:functions_dictionary[i](headers_block[0].split('- ')[1:][0].split('line:')[1],lines_separators[i]) 
 for i in lines_separators}

In [ ]:
'''
{i:functions_dictionary[i](j.split('line:')[1],lines_separators[i]) 
 for i in lines_separators}
 '''

In [ ]:


testing_lines2 = [{'header': i.split('\n')[0].strip(),
                               'lines': [
                                           {k:functions_dictionary[k](j.split('line:')[1],lines_separators[k]) 
                                             for k in lines_separators}
                                            for j in i.split('- ')[1:]
                                        ] 
 } for i in headers_block]

In [ ]:
testing_lines2[0]

In [ ]:
len(episode_data['transcript']),len(episode_data['transcript'][0]['lines']),len(episode_data['transcript'][0]['lines'][0]),

In [ ]:
episode_data['transcript'][0]['lines'][0]

In [ ]:
testing_lines2[0]['lines'][0]

In [ ]:
episode_data['transcript'][0]['lines'][4]==testing_lines2[0]['lines'][4]

In [ ]:
episode_data['transcript']==testing_lines2

In [ ]:
out1 = clean_markdown(text)

In [ ]:
out1.keys()

In [ ]:
episode_data.keys()

In [ ]:
out1==episode_data

# save json

In [ ]:
episode_json_path = os.path.join(data_path,"episode.json")

In [ ]:
json_str = json.dumps(out1, indent=4)
with open(episode_json_path, "w",encoding="utf-8") as f:
    f.write(json_str)